# Notebook 8: Add HNSW Edges

This notebook constructs an HNSW index from node embeddings to quickly approximate the K-nearest neighbors (KNN) for each node.

Each node is then linked with its **8 nearest neighbors**. Neighbors with a **similarity ≤ 0.6** are skipped.

## HNSW Hyperparameters
- **M** = 32  
- **efConstruction** = 200  
- **efSearch** = 64  

## Output
- The resulting **HNSW index** is saved.  
- The resulting graph is saved as **G5 - semantically enriched graph**.

In [1]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

The directory of this script is: c:\Users\HP\Desktop\Projects\NodeRAG\graphs
The root directory is: c:\Users\HP\Desktop\Projects\NodeRAG


In [2]:
import sys
sys.path.append(root_path)
from graphs.Node import Node

In [3]:
import pickle
with open(f"{root_path}/graphs/data/graphs/G4_text_inserted_graph.pkl", "rb") as f:
    medical_g4 = pickle.load(f)

In [4]:
#sanity check
for node_id in medical_g4:
    node = medical_g4[node_id]
    for edge in node.edges:
        if node.edges[edge] > 1:
            print(node_id,"-",edge,"-",node.edges[edge])
print("-"*40)
for node_id in medical_g4:
    node = medical_g4[node_id]
    for edge in node.edges:
        if edge == node_id:
            print(node_id)
print("-"*40)
#calculate total edge weight
sum_edge = 0
for node_id in medical_g4:
    node = medical_g4[node_id]
    for edge in node.edges:
        sum_edge += node.edges[edge]
print("Total edge weight in G4:", sum_edge)
print("-"*40)

----------------------------------------
----------------------------------------
Total edge weight in G4: 215004
----------------------------------------


In [5]:
import json
import faiss
import numpy as np
#embeddings of S,A,H,T nodes

#load faiss index file
index = faiss.read_index("data/embedding/medical_index.faiss")
with open("data/embedding/medical_ids.json", "r") as f:
    medical_embedding_ids = json.load(f)
index.ntotal, len(medical_embedding_ids)

#reconstruct vectors from faiss index
num_vectors = index.ntotal
dimension = index.d
embeddings = np.zeros((num_vectors, dimension), dtype='float32')
for i in range(num_vectors):
    embeddings[i] = index.reconstruct(i)

print("Embeddings shape:", embeddings.shape)
embeddings

Embeddings shape: (4426, 768)


array([[-0.06428347, -0.05439772, -0.04260492, ..., -0.0463121 ,
        -0.03141087,  0.0049261 ],
       [-0.04751787, -0.05777418, -0.02587977, ..., -0.04238609,
        -0.00359848,  0.0272001 ],
       [-0.06036756, -0.067439  , -0.05656344, ..., -0.06152818,
        -0.01679316,  0.01591949],
       ...,
       [ 0.04048249, -0.0287718 , -0.03389297, ..., -0.00295302,
         0.0417009 ,  0.00099732],
       [ 0.00630154, -0.0139234 , -0.01267361, ..., -0.09236696,
         0.01060561, -0.01778929],
       [ 0.01245843, -0.01854508, -0.01106029, ..., -0.08220403,
         0.01130273,  0.05532239]], dtype=float32)

In [6]:
import time
import random
#build hnsw index
start_time = time.time()
M = 32
hnsw = faiss.IndexHNSWFlat(embeddings.shape[1], M, faiss.METRIC_INNER_PRODUCT)
hnsw.hnsw.efConstruction = 200
hnsw.hnsw.efSearch = 64
hnsw.add(embeddings)
faiss.write_index(hnsw, "data/embedding/medical_index_hnsw.faiss")
end_time = time.time()
print(f"HNSW index built in {end_time - start_time:.2f} seconds.")

#test

k_test = 8
q = embeddings[random.randint(0,embeddings.shape[0]-1)].reshape(1, -1)
distance, idx = hnsw.search(q, k_test)
for i in range(k_test):
    print(f"Neighbor {medical_embedding_ids[idx[0][i]]}: Index={idx[0][i]}, Similarity={distance[0][i]}")

HNSW index built in 0.21 seconds.
Neighbor medical-N-361-A-0: Index=3147, Similarity=0.9999998807907104
Neighbor medical-N-175-A-0: Index=3079, Similarity=0.8450511693954468
Neighbor medical-108-S-0: Index=587, Similarity=0.7574504613876343
Neighbor medical-120-S-1: Index=658, Similarity=0.7477802038192749
Neighbor medical-204-S-0: Index=1114, Similarity=0.732801079750061
Neighbor medical-523-S-3: Index=2834, Similarity=0.7293275594711304
Neighbor medical-N-177-A-0: Index=3080, Similarity=0.7215619087219238
Neighbor medical-11-S-1: Index=51, Similarity=0.7195706367492676


In [7]:
#add hnsw edges to graph
added_semantic_edges = set()

for i in range(len(medical_embedding_ids)):
    node_id = medical_embedding_ids[i]
    node = medical_g4[node_id]
    node_embedding = embeddings[i].reshape(1,-1)
    k = 16
    similarities, indices = hnsw.search(node_embedding, k+1)
    for j in range(k+1):
        neighbor_id = medical_embedding_ids[indices[0][j]]
        if neighbor_id == node_id:
            continue
        neighbor = medical_g4[neighbor_id]
        similarity = round(similarities[0][j], 3)
        if similarity <= 0.6:
            continue

        edge_key = frozenset((node_id, neighbor_id))
        if edge_key in added_semantic_edges:
            continue
        added_semantic_edges.add(edge_key)

        node.link(neighbor, similarity)
        neighbor.link(node, similarity)

In [8]:
#calculate total edge weight after adding hnsw edges
sum_edge = 0
for node_id in medical_g4:
    node = medical_g4[node_id]
    for edge in node.edges:
        sum_edge += node.edges[edge]
print("Total edge weight:", sum_edge)
print("-"*40)
for node_id in medical_g4:
    node = medical_g4[node_id]
    for edge in node.edges:
        if edge == node_id:
            print(node_id)

Total edge weight: 270701.6840170622
----------------------------------------


In [9]:
with open(f"{root_path}/graphs/data/graphs/G5_semantically_enriched_graph.pkl", "wb") as f:
    pickle.dump(medical_g4, f)
with open(f"{root_path}/graphs/data/graphs/G5_semantically_enriched_graph.pkl", "rb") as f:
    medical_g5 = pickle.load(f)

In [10]:
#check
keys = list(medical_g5.keys())
for i in range(999):
    node_id = keys[i]
    node = medical_g5[node_id]
    for j in range(i+1, 1000):
        neighbor_id = keys[j]
        if neighbor_id in node.edges:
            print(node_id, "-", neighbor_id, "-", node.edges[neighbor_id])

medical-0-S-0 - medical-0-S-1 - 0.774
medical-0-S-0 - medical-1-S-2 - 0.666
medical-0-S-0 - medical-1-S-3 - 0.63
medical-0-S-0 - medical-2-S-1 - 0.725
medical-0-S-0 - medical-13-S-0 - 0.695
medical-0-S-1 - medical-0-S-2 - 0.712
medical-0-S-1 - medical-1-S-0 - 0.62
medical-0-S-1 - medical-1-S-2 - 0.763
medical-0-S-1 - medical-1-S-3 - 0.666
medical-0-S-1 - medical-1-S-4 - 0.7
medical-0-S-1 - medical-2-S-0 - 0.682
medical-0-S-1 - medical-2-S-1 - 0.751
medical-0-S-1 - medical-3-S-0 - 0.618
medical-0-S-1 - medical-13-S-0 - 0.732
medical-0-S-1 - medical-13-S-1 - 0.616
medical-0-S-1 - medical-13-S-4 - 0.656
medical-0-S-2 - medical-1-S-2 - 0.686
medical-0-S-3 - medical-1-S-0 - 0.767
medical-0-S-3 - medical-13-S-3 - 0.878
medical-0-S-3 - medical-13-S-4 - 0.668
medical-0-S-3 - medical-14-S-0 - 0.678
medical-1-S-0 - medical-13-S-3 - 0.625
medical-1-S-0 - medical-13-S-4 - 0.804
medical-1-S-0 - medical-14-S-0 - 0.738
medical-1-S-1 - medical-3-S-3 - 0.63
medical-1-S-1 - medical-4-S-0 - 0.638
medical